# Part 2: Independent Fine-tuning on New Dataset

This notebook handles:
- Loading a new dataset (Part 2) with automatic column detection
- Testing the pretrained model on the new dataset (baseline)
- Fine-tuning the pretrained model on the new dataset
- Comprehensive evaluation and comparison of pretrained vs fine-tuned performance

**Workflow**: Pretrained Model → Test on Part 2 → Fine-tune on Part 2 → Compare Results

**Features**: Flexible dataset handling, automatic column detection, robust preprocessing

In [ ]:
# Install dependencies (run once)
!pip install -U transformers datasets evaluate accelerate scikit-learn seaborn >/dev/null 2>&1 || true
print("Dependencies installed successfully.")

In [ ]:
# Configuration - Update these for your dataset - GPU OPTIMIZED
MODEL_NAME = "protectai/deberta-v3-base-prompt-injection"
PART2_FINETUNED_DIR = './models/hf_deberta_part2_finetuned'  # New model for Part 2

# PART 2 DATASET CONFIGURATION
# Option 1: Hugging Face dataset ID
PART2_DATASET_ID = 'reshabhs/SPML_Chatbot_Prompt_Injection'  # e.g., "deepset/prompt-injections"

# Option 2: Local dataset path
PART2_LOCAL_PATH = None  # e.g., "./data/my_dataset.csv"

# Training control - set to True when ready to fine-tune on Part 2 dataset
DO_PART2_TRAIN = True  # GPU READY

# For quick testing - GPU can handle full dataset
EVAL_SAMPLE_SIZE = None  # Using full dataset for GPU training

print("🚀 Part 2 GPU Configuration set.")
print(f"Base pretrained model: {MODEL_NAME}")
print(f"Part 2 fine-tuned model will be saved to: {PART2_FINETUNED_DIR}")
print(f"Part 2 dataset: {PART2_DATASET_ID or PART2_LOCAL_PATH or 'Not specified - will use fallback'}")
print(f"Part 2 training enabled: {DO_PART2_TRAIN}")
print("🔥 GPU training mode - using full dataset for Part 2!")

In [ ]:
# Import libraries and setup
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import os
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve, average_precision_score
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

# Fix tokenizer parallelism warning
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Device setup and memory management
import gc
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Clear GPU memory if available
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print("🔧 GPU memory cleared and ready for Part 2")

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LENGTH = 256

In [ ]:
# Flexible dataset loading with automatic column detection
def load_part2_dataset(dataset_id=None, local_path=None, fallback_dataset="deepset/prompt-injections"):
    """
    Load Part 2 dataset with flexible column detection
    """
    
    def detect_columns(dataset_sample):
        """Auto-detect text and label columns"""
        columns = list(dataset_sample.keys())
        print(f"Available columns: {columns}")
        
        # Common text column names
        text_candidates = ['text', 'prompt', 'input', 'query', 'content', 'message', 'sentence']
        text_col = None
        for candidate in text_candidates:
            if candidate in columns:
                text_col = candidate
                break
        
        if not text_col:
            # Use first string column
            for col in columns:
                if isinstance(dataset_sample[col], str):
                    text_col = col
                    break
        
        # Common label column names
        label_candidates = ['label', 'labels', 'target', 'class', 'classification', 'category', 'is_injection']
        label_col = None
        for candidate in label_candidates:
            if candidate in columns:
                label_col = candidate
                break
        
        if not label_col:
            # Use first numeric column
            for col in columns:
                if isinstance(dataset_sample[col], (int, float, bool)):
                    label_col = col
                    break
        
        print(f"Detected text column: {text_col}")
        print(f"Detected label column: {label_col}")
        
        return text_col, label_col
    
    def normalize_labels(labels):
        """Convert labels to binary 0/1 format"""
        unique_labels = list(set(labels))
        print(f"Unique labels found: {unique_labels}")
        
        # If already binary
        if set(unique_labels).issubset({0, 1, True, False}):
            return [int(label) for label in labels]
        
        # String labels - map to binary
        safe_keywords = ['safe', 'benign', 'normal', 'legitimate', 'clean', 'no', 'false', '0']
        injection_keywords = ['injection', 'attack', 'malicious', 'harmful', 'yes', 'true', '1']
        
        binary_labels = []
        for label in labels:
            label_str = str(label).lower().strip()
            if any(keyword in label_str for keyword in safe_keywords):
                binary_labels.append(0)
            elif any(keyword in label_str for keyword in injection_keywords):
                binary_labels.append(1)
            else:
                # Default mapping: first unique → 0, second unique → 1
                binary_labels.append(unique_labels.index(label) % 2)
        
        print(f"Label mapping: {dict(zip(unique_labels, set(binary_labels)))}")
        return binary_labels
    
    # Try loading dataset
    dataset = None
    source_info = ""
    
    if dataset_id:
        try:
            print(f"Loading dataset from Hugging Face: {dataset_id}")
            dataset = load_dataset(dataset_id)
            source_info = f"Hugging Face: {dataset_id}"
        except Exception as e:
            print(f"Failed to load {dataset_id}: {e}")
    
    elif local_path:
        try:
            print(f"Loading local dataset: {local_path}")
            if local_path.endswith('.csv'):
                df = pd.read_csv(local_path)
            elif local_path.endswith('.json'):
                df = pd.read_json(local_path)
            else:
                df = pd.read_csv(local_path)  # Try CSV as default
            
            dataset = Dataset.from_pandas(df)
            dataset = {'test': dataset}  # Treat as test set
            source_info = f"Local file: {local_path}"
        except Exception as e:
            print(f"Failed to load local file {local_path}: {e}")
    
    # Fallback dataset
    if dataset is None:
        print(f"Using fallback dataset: {fallback_dataset}")
        try:
            dataset = load_dataset(fallback_dataset)
            source_info = f"Fallback: {fallback_dataset}"
        except Exception as e:
            print(f"Failed to load fallback dataset: {e}")
            return None, None, None
    
    # Get test split
    test_split = None
    for split_name in ['test', 'validation', 'val', 'train']:
        if split_name in dataset:
            test_split = dataset[split_name]
            print(f"Using '{split_name}' split for evaluation")
            break
    
    if test_split is None:
        print("No suitable split found")
        return None, None, None
    
    print(f"Dataset loaded: {len(test_split)} samples")
    print(f"Sample: {test_split[0]}")
    
    # Auto-detect columns
    text_col, label_col = detect_columns(test_split[0])
    
    if not text_col or not label_col:
        print(f"Could not detect required columns. Available: {list(test_split[0].keys())}")
        return None, None, None
    
    # Extract and normalize data
    texts = [item[text_col] for item in test_split]
    labels = normalize_labels([item[label_col] for item in test_split])
    
    # Create standardized dataset
    standardized_data = {
        'text': texts,
        'labels': labels  # Note: 'labels' not 'label'
    }
    
    processed_dataset = Dataset.from_dict(standardized_data)
    
    print(f"\nProcessed dataset:")
    print(f"Samples: {len(processed_dataset)}")
    print(f"Label distribution: {dict(pd.Series(labels).value_counts().sort_index())}")
    print(f"Sample texts (first 3):")
    for i in range(min(3, len(texts))):
        preview = texts[i][:100] + "..." if len(texts[i]) > 100 else texts[i]
        print(f"  {i+1}. [{labels[i]}] {preview}")
    
    return processed_dataset, source_info, {'text_col': text_col, 'label_col': label_col}

# Load Part 2 dataset
part2_dataset, dataset_source, column_info = load_part2_dataset(
    dataset_id=PART2_DATASET_ID,
    local_path=PART2_LOCAL_PATH
)

if part2_dataset is None:
    print("❌ Failed to load Part 2 dataset")
else:
    print(f"✅ Part 2 dataset loaded successfully")
    print(f"Source: {dataset_source}")

In [ ]:
# Tokenize Part 2 dataset
if part2_dataset is not None:
    print("Tokenizing Part 2 dataset...")
    
    def preprocess_part2(examples):
        return tokenizer(
            examples['text'], 
            truncation=True, 
            padding='max_length', 
            max_length=MAX_LENGTH
        )
    
    tokenized_part2 = part2_dataset.map(preprocess_part2, batched=True)
    tokenized_part2.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    
    print(f"Tokenization complete. Sample:")
    sample = tokenized_part2[0]
    print(f"Input IDs shape: {sample['input_ids'].shape}")
    print(f"Attention mask shape: {sample['attention_mask'].shape}")
    print(f"Label: {sample['labels']}")
else:
    print("Skipping tokenization - no dataset loaded")
    tokenized_part2 = None

In [ ]:
# PART 2 GPU TRAINING CONFIGURATION
# High-performance settings for Part 2 generalization testing

PART2_TRAINING_INTENSITY = "GPU_BEAST"  # Options: "QUICK", "BALANCED", "MAXIMUM", "EXTREME", "GPU_BEAST"

# Part 2 GPU-optimized configurations
PART2_TRAINING_CONFIGS = {
    "QUICK": {
        "epochs": 3,
        "batch_size": 16,
        "learning_rate": 2e-5,
        "patience": 2,
        "description": "Fast Part 2 training"
    },
    "BALANCED": {
        "epochs": 8,  # Good for generalization testing
        "batch_size": 24,
        "learning_rate": 2.5e-5,
        "patience": 3,
        "description": "Balanced Part 2 training"
    },
    "MAXIMUM": {
        "epochs": 12,  # Strong generalization
        "batch_size": 24,  # Reduced for stability
        "learning_rate": 3e-5,
        "patience": 4,
        "description": "High accuracy Part 2 training"
    },
    "EXTREME": {
        "epochs": 18,  # Excellent generalization
        "batch_size": 28,  # Reduced for stability
        "learning_rate": 3e-5,
        "patience": 5,
        "description": "Maximum Part 2 accuracy"
    },
    "GPU_BEAST": {
        "epochs": 20,  # Stable generalization testing
        "batch_size": 32,  # Reduced for stability
        "learning_rate": 2.5e-5,
        "patience": 6,
        "description": "Stable GPU beast mode for Part 2 - maximum generalization"
    }
}

# Apply Part 2 configuration
part2_config = PART2_TRAINING_CONFIGS[PART2_TRAINING_INTENSITY]
print(f"🚀 Part 2 Training Intensity: {PART2_TRAINING_INTENSITY}")
print(f"Configuration: {part2_config['description']}")
print(f"• Epochs: {part2_config['epochs']}")
print(f"• Batch size: {part2_config['batch_size']} (GPU optimized)")
print(f"• Learning rate: {part2_config['learning_rate']}")
print(f"• Early stopping patience: {part2_config['patience']} epochs")

# Part 2 GPU estimates
part2_time_estimates = {
    "QUICK": "2-4 minutes on good GPU",
    "BALANCED": "6-12 minutes on good GPU",
    "MAXIMUM": "10-20 minutes on good GPU", 
    "EXTREME": "15-30 minutes on good GPU",
    "GPU_BEAST": "20-40 minutes on good GPU"
}
print(f"Estimated Part 2 training time: {part2_time_estimates[PART2_TRAINING_INTENSITY]}")

print(f"\n🎯 Part 2 Generalization Benefits:")
print(f"• Tests model on completely different dataset")
print(f"• Validates training approach across domains")
print(f"• {part2_config['epochs']} epochs for strong generalization")
print(f"• GPU acceleration for fast results")
print(f"• Independent fine-tuning from scratch")

In [ ]:
# 🔧 PART 2 GPU TRAINING FIX - Run this cell if you get gradient errors
# This cell fixes the "backward through the graph a second time" error for Part 2

print("🔧 Applying Part 2 GPU training fixes...")

# Clear GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✅ GPU memory cleared for Part 2")

# Reset Part 2 training configuration with stable settings
STABLE_PART2_CONFIG = {
    "epochs": 16,  # Excellent for generalization, stable
    "batch_size": 20,  # Safe batch size for most GPUs
    "learning_rate": 2.5e-5,
    "patience": 5,
    "description": "Stable Part 2 GPU training - excellent generalization with reliability"
}

# Update the GPU_BEAST config to be more stable for Part 2
PART2_TRAINING_CONFIGS["GPU_BEAST"] = STABLE_PART2_CONFIG

print("✅ Part 2 training configuration updated for stability:")
print(f"• Epochs: {STABLE_PART2_CONFIG['epochs']} (excellent for generalization)")
print(f"• Batch size: {STABLE_PART2_CONFIG['batch_size']} (memory-safe)")
print(f"• Gradient checkpointing: DISABLED (prevents errors)")
print(f"• Workers: 4 (stable)")

print("\n🚀 Part 2 ready to restart training with stable settings!")
print("   Re-run the Part 2 training cell now.")
print("   This will test generalization on a completely different dataset!")

In [ ]:
# Comprehensive evaluation function (same as Part 1)
def comprehensive_evaluation(model_path, dataset, dataset_name="Test Dataset", sample_size=None):
    """Comprehensive model evaluation with detailed metrics and visualizations"""
    
    print(f"\n{'='*60}")
    print(f"COMPREHENSIVE EVALUATION: {dataset_name}")
    print(f"Model: {model_path}")
    print(f"{'='*60}")
    
    # Load model
    try:
        if isinstance(model_path, str) and os.path.isdir(model_path):
            model_eval = AutoModelForSequenceClassification.from_pretrained(model_path)
        else:
            model_eval = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
        model_eval.to(device)
        model_eval.eval()
        print("Model loaded successfully")
    except Exception as e:
        print(f"Failed to load model: {e}")
        return None
    
    # Prepare data
    test_ds = dataset
    if sample_size:
        indices = list(range(min(sample_size, len(test_ds))))
        test_ds = test_ds.select(indices)
    
    loader = DataLoader(test_ds, batch_size=32)
    
    # Run detailed inference
    preds, probs, refs = [], [], []
    loss_fn = torch.nn.CrossEntropyLoss()
    total_loss, batches = 0.0, 0
    
    print("Running inference...")
    for batch in tqdm(loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        with torch.no_grad():
            outputs = model_eval(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            
            # Calculate loss
            loss = loss_fn(logits, labels)
            total_loss += loss.item()
            batches += 1
            
            # Get predictions and probabilities
            prob = torch.softmax(logits, dim=-1).cpu().numpy()
            pred = np.argmax(prob, axis=-1).tolist()
            
        preds.extend(pred)
        probs.extend(prob.tolist())
        refs.extend(labels.cpu().numpy().tolist())
    
    # Calculate metrics
    n_samples = len(refs)
    avg_loss = total_loss / batches
    accuracy = accuracy_score(refs, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(refs, preds, average='binary', zero_division=0)
    cm = confusion_matrix(refs, preds)
    report = classification_report(refs, preds, target_names=['Safe', 'Prompt Injection'], zero_division=0, output_dict=True)
    
    # Print results
    print(f"\nRESULTS SUMMARY:")
    print(f"Test Samples: {n_samples:,}")
    print(f"Test Loss: {avg_loss:.4f}")
    print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
    print(f"Recall: {recall:.4f} ({recall*100:.2f}%)")
    print(f"F1 Score: {f1:.4f} ({f1*100:.2f}%)")
    
    # Confusion matrix details
    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
        print(f"\nCONFUSION MATRIX BREAKDOWN:")
        print(f"True Negatives (Safe): {tn:,}")
        print(f"False Positives (Safe → Injection): {fp:,}")
        print(f"False Negatives (Injection → Safe): {fn:,}")
        print(f"True Positives (Injection): {tp:,}")
    
    # Visualizations
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(f'Model Evaluation Results - {dataset_name}', fontsize=16, fontweight='bold')
    
    # 1. Confusion Matrix
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Safe', 'Prompt Injection'],
                yticklabels=['Safe', 'Prompt Injection'],
                ax=axes[0,0])
    axes[0,0].set_title('Confusion Matrix')
    axes[0,0].set_xlabel('Predicted')
    axes[0,0].set_ylabel('True')
    
    # 2. ROC Curve
    probs_array = np.array(probs)
    if probs_array.shape[1] == 2:
        pos_probs = probs_array[:, 1]
        fpr, tpr, _ = roc_curve(refs, pos_probs)
        roc_auc = auc(fpr, tpr)
        
        axes[0,1].plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.3f})', color='darkorange', lw=2)
        axes[0,1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', alpha=0.8)
        axes[0,1].set_xlim([0.0, 1.0])
        axes[0,1].set_ylim([0.0, 1.05])
        axes[0,1].set_xlabel('False Positive Rate')
        axes[0,1].set_ylabel('True Positive Rate')
        axes[0,1].set_title('ROC Curve')
        axes[0,1].legend(loc="lower right")
        axes[0,1].grid(True, alpha=0.3)
        
        # 3. Precision-Recall Curve
        precision_curve, recall_curve, _ = precision_recall_curve(refs, pos_probs)
        ap = average_precision_score(refs, pos_probs)
        
        axes[1,0].plot(recall_curve, precision_curve, label=f'PR Curve (AP = {ap:.3f})', color='darkgreen', lw=2)
        axes[1,0].set_xlabel('Recall')
        axes[1,0].set_ylabel('Precision')
        axes[1,0].set_title('Precision-Recall Curve')
        axes[1,0].legend(loc="lower left")
        axes[1,0].grid(True, alpha=0.3)
    
    # 4. Metrics Summary
    metrics_data = {
        'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC'],
        'Score': [accuracy, precision, recall, f1, roc_auc if 'roc_auc' in locals() else 0],
        'Percentage': [f"{accuracy*100:.1f}%", f"{precision*100:.1f}%", 
                      f"{recall*100:.1f}%", f"{f1*100:.1f}%", 
                      f"{roc_auc*100:.1f}%" if 'roc_auc' in locals() else "N/A"]
    }
    
    axes[1,1].axis('tight')
    axes[1,1].axis('off')
    table = axes[1,1].table(cellText=[[row[0], f"{row[1]:.4f}", row[2]] for row in zip(metrics_data['Metric'], metrics_data['Score'], metrics_data['Percentage'])],
                           colLabels=['Metric', 'Score', 'Percentage'],
                           cellLoc='center',
                           loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)
    axes[1,1].set_title('Performance Summary')
    
    plt.tight_layout()
    plt.show()
    
    # Show sample predictions with errors
    print(f"\nSAMPLE PREDICTIONS:")
    error_indices = [i for i, (pred, ref) in enumerate(zip(preds, refs)) if pred != ref]
    correct_indices = [i for i, (pred, ref) in enumerate(zip(preds, refs)) if pred == ref]
    
    print(f"\nCorrect Predictions (sample):")
    for i, idx in enumerate(correct_indices[:3]):
        text_preview = part2_dataset[idx]['text'][:100] + "..." if len(part2_dataset[idx]['text']) > 100 else part2_dataset[idx]['text']
        print(f"  ✅ True: {refs[idx]}, Pred: {preds[idx]} | {text_preview}")
    
    if error_indices:
        print(f"\nIncorrect Predictions (sample):")
        for i, idx in enumerate(error_indices[:3]):
            text_preview = part2_dataset[idx]['text'][:100] + "..." if len(part2_dataset[idx]['text']) > 100 else part2_dataset[idx]['text']
            print(f"  ❌ True: {refs[idx]}, Pred: {preds[idx]} | {text_preview}")
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'loss': avg_loss,
        'auc_roc': roc_auc if 'roc_auc' in locals() else None
    }

print("Evaluation function ready.")

In [ ]:
# Evaluate pretrained model on Part 2 dataset (BASELINE)
if tokenized_part2 is not None:
    print("🔍 Evaluating PRETRAINED model on Part 2 dataset (BASELINE)...")
    pretrained_part2_results = comprehensive_evaluation(
        MODEL_NAME, 
        tokenized_part2, 
        f"Part 2 Dataset (Pretrained Baseline) - {dataset_source}", 
        EVAL_SAMPLE_SIZE
    )
else:
    print("⚠️ Skipping pretrained evaluation - no Part 2 dataset loaded")
    pretrained_part2_results = None

In [ ]:
# Fine-tune pretrained model on Part 2 dataset
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback
import evaluate
from collections import Counter

# Coerce DO_PART2_TRAIN to boolean
if isinstance(DO_PART2_TRAIN, str):
    DO_PART2_TRAIN = DO_PART2_TRAIN.strip().lower() in ('true', '1', 'yes')
else:
    DO_PART2_TRAIN = bool(DO_PART2_TRAIN)

if DO_PART2_TRAIN and tokenized_part2 is not None:
    print("🚀 Starting fine-tuning on Part 2 dataset...")
    
    # Create models directory
    os.makedirs('./models', exist_ok=True)
    
    # Load fresh pretrained model for training
    model_to_train = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
    
    # Compute metrics function
    metric_acc = evaluate.load('accuracy')
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        acc = metric_acc.compute(predictions=preds, references=labels)
        precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
        return {
            'accuracy': acc['accuracy'], 
            'precision': float(precision), 
            'recall': float(recall), 
            'f1': float(f1)
        }
    
    # Data collator
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    # Class weights for imbalanced data
    def make_class_weights(labels):
        if torch.is_tensor(labels):
            labels = labels.cpu().numpy().tolist()
        
        counts = Counter(labels)
        n_total = sum(counts.values())
        num_classes = max(counts.keys()) + 1
        
        weights = []
        for i in range(num_classes):
            if i in counts:
                weight = n_total / (num_classes * counts[i])
            else:
                weight = 1.0
            weights.append(weight)
        
        print(f"Part 2 class distribution: {dict(counts)}")
        print(f"Part 2 class weights: {weights}")
        return torch.tensor(weights, dtype=torch.float)
    
    # Custom Trainer with class weights
    class WeightedTrainer(Trainer):
        def __init__(self, *args, class_weights=None, **kwargs):
            super().__init__(*args, **kwargs)
            self.class_weights = class_weights
        
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.get('labels')
            outputs = model(**inputs)
            logits = outputs.get('logits')
            
            if labels is not None:
                if self.class_weights is not None:
                    loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
                else:
                    loss_fct = torch.nn.CrossEntropyLoss()
                loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
            else:
                loss = None
            
            return (loss, outputs) if return_outputs else loss
    
    # Split Part 2 dataset for training/validation
    print("Splitting Part 2 dataset for training...")
    train_test_split = tokenized_part2.train_test_split(test_size=0.2, seed=42)
    train_dataset = train_test_split['train']
    eval_dataset = train_test_split['test']
    
    print(f"Training samples: {len(train_dataset)}")
    print(f"Evaluation samples: {len(eval_dataset)}")
    
    # Compute class weights
    try:
        train_labels = train_dataset['labels']
        if torch.is_tensor(train_labels):
            train_labels = train_labels.cpu().numpy()
        class_weights = make_class_weights(train_labels)
    except Exception as e:
        print(f"Could not compute class weights: {e}")
        class_weights = None
    
    # GPU-optimized training arguments for Part 2
    try:
        selected_part2_config = PART2_TRAINING_CONFIGS[PART2_TRAINING_INTENSITY]
        print(f"\nUsing {PART2_TRAINING_INTENSITY} Part 2 configuration:")
        print(f"• {selected_part2_config['description']}")
    except NameError:
        print("Using default BALANCED Part 2 configuration")
        selected_part2_config = {
            "epochs": 8,
            "batch_size": 24,
            "learning_rate": 2.5e-5,
            "patience": 3
        }
    
    training_args = TrainingArguments(
        output_dir=PART2_FINETUNED_DIR,
        num_train_epochs=selected_part2_config['epochs'],
        per_device_train_batch_size=selected_part2_config['batch_size'],
        per_device_eval_batch_size=40,  # Larger eval batch for GPU
        eval_accumulation_steps=1,
        eval_strategy='epoch',
        save_strategy='epoch',
        logging_strategy='steps',
        logging_steps=max(10, selected_part2_config['epochs'] * 3),
        learning_rate=selected_part2_config['learning_rate'],
        weight_decay=0.01,
        warmup_ratio=0.12,
        load_best_model_at_end=True,
        metric_for_best_model='f1',  # F1 better for imbalanced data
        greater_is_better=True,
        save_total_limit=3,
        fp16=torch.cuda.is_available(),  # GPU FP16
        gradient_accumulation_steps=1,  # No accumulation needed with large batches
        dataloader_num_workers=4,  # Reduced workers for stability
        gradient_checkpointing=False,  # Disabled to prevent gradient graph errors
        report_to=None  # Disable wandb/tensorboard
    )
    
    # Create trainer
    try:
        trainer = WeightedTrainer(
            model=model_to_train,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            tokenizer=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
            class_weights=class_weights
        )
        print("Using WeightedTrainer with class balancing")
    except Exception as e:
        print(f"WeightedTrainer failed ({e}), using standard Trainer")
        trainer = Trainer(
            model=model_to_train,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            tokenizer=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
        )
    
    # Add early stopping with configured patience
    trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=selected_part2_config['patience']))
    
    print(f"\n{'='*60}")
    print(f"STARTING PART 2 {PART2_TRAINING_INTENSITY} TRAINING")
    print(f"{'='*60}")
    print(f"• Epochs: {selected_part2_config['epochs']}")
    print(f"• Batch size: {selected_part2_config['batch_size']}")
    print(f"• Learning rate: {selected_part2_config['learning_rate']}")
    print(f"• Early stopping patience: {selected_part2_config['patience']} epochs")
    print(f"• Dataset: Different from Part 1 (generalization test)")
    print(f"🚀 GPU-accelerated Part 2 training started...")
    
    # Clear GPU memory before training
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("🔧 GPU memory cleared for Part 2 training")
    
    try:
        train_result = trainer.train()
    except RuntimeError as e:
        if "backward through the graph a second time" in str(e):
            print("⚠️ Gradient checkpointing conflict in Part 2. Restarting with safer settings...")
            # Disable gradient checkpointing and try again
            training_args.gradient_checkpointing = False
            trainer = WeightedTrainer(
                model=model_to_train,
                args=training_args,
                train_dataset=train_dataset,
                eval_dataset=eval_dataset,
                tokenizer=tokenizer,
                data_collator=data_collator,
                compute_metrics=compute_metrics,
                class_weights=class_weights
            )
            trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=selected_part2_config['patience']))
            train_result = trainer.train()
        else:
            raise e
    
    # Save model and training logs
    trainer.save_model(PART2_FINETUNED_DIR)
    
    try:
        with open(os.path.join(PART2_FINETUNED_DIR, 'part2_train_log.json'), 'w') as f:
            json.dump(trainer.state.log_history, f, default=str, indent=2)
        print(f"Training logs saved to {PART2_FINETUNED_DIR}/part2_train_log.json")
    except Exception as e:
        print(f"Could not save training logs: {e}")
    
    print(f"\n{'='*60}")
    print(f"PART 2 FINE-TUNING COMPLETED!")
    print(f"{'='*60}")
    print(f"Model saved to: {PART2_FINETUNED_DIR}")
    print(f"Best metric: {trainer.state.best_metric}")
    print(f"Training time: {train_result.metrics.get('train_runtime', 'Unknown')} seconds")
    
elif tokenized_part2 is None:
    print("⚠️ No Part 2 dataset loaded - cannot fine-tune")
else:
    if os.path.isdir(PART2_FINETUNED_DIR):
        print(f"DO_PART2_TRAIN is False. Found existing Part 2 model at {PART2_FINETUNED_DIR}")
    else:
        print(f"DO_PART2_TRAIN is False. Set DO_PART2_TRAIN = True to start training on Part 2 dataset.")
        print(f"Model will be saved to: {PART2_FINETUNED_DIR}")

In [ ]:
# Evaluate Part 2 fine-tuned model
if tokenized_part2 is not None and os.path.isdir(PART2_FINETUNED_DIR):
    print("🎯 Evaluating PART 2 FINE-TUNED model...")
    part2_finetuned_results = comprehensive_evaluation(
        PART2_FINETUNED_DIR, 
        tokenized_part2, 
        f"Part 2 Dataset (Fine-tuned on Part 2) - {dataset_source}", 
        EVAL_SAMPLE_SIZE
    )
    
    # Save results to Part 2 fine-tuned model directory
    if part2_finetuned_results:
        try:
            results_summary = {
                'part2_dataset_source': dataset_source,
                'part2_finetuned_evaluation': part2_finetuned_results,
                'column_mapping': column_info,
                'evaluation_samples': EVAL_SAMPLE_SIZE,
                'model_path': PART2_FINETUNED_DIR
            }
            
            filepath = os.path.join(PART2_FINETUNED_DIR, 'part2_finetuned_results.json')
            with open(filepath, 'w') as f:
                json.dump(results_summary, f, indent=2)
            print(f"\n💾 Part 2 fine-tuned results saved to: {filepath}")
        except Exception as e:
            print(f"Could not save Part 2 fine-tuned results: {e}")
            
elif tokenized_part2 is not None:
    print(f"⚠️ Part 2 fine-tuned model not found at {PART2_FINETUNED_DIR}")
    print("Set DO_PART2_TRAIN = True and run the training cell first.")
    part2_finetuned_results = None
else:
    print("⚠️ Skipping Part 2 fine-tuned evaluation - no Part 2 dataset loaded")
    part2_finetuned_results = None

In [ ]:
# Performance comparison: Pretrained vs Part 2 Fine-tuned
if pretrained_part2_results and part2_finetuned_results:
    print(f"\n{'='*80}")
    print(f"PERFORMANCE COMPARISON - PART 2 FINE-TUNING EFFECTIVENESS")
    print(f"{'='*80}")
    print(f"Dataset: {dataset_source}")
    print(f"Evaluation samples: {EVAL_SAMPLE_SIZE or 'All'}")
    print(f"Comparison: Pretrained Model vs Model Fine-tuned on Part 2 Dataset")
    
    # Create comparison table
    comparison_data = {
        'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Loss'],
        'Pretrained (Baseline)': [
            f"{pretrained_part2_results['accuracy']:.4f} ({pretrained_part2_results['accuracy']*100:.2f}%)",
            f"{pretrained_part2_results['precision']:.4f} ({pretrained_part2_results['precision']*100:.2f}%)",
            f"{pretrained_part2_results['recall']:.4f} ({pretrained_part2_results['recall']*100:.2f}%)",
            f"{pretrained_part2_results['f1']:.4f} ({pretrained_part2_results['f1']*100:.2f}%)",
            f"{pretrained_part2_results['loss']:.4f}"
        ],
        'Part 2 Fine-tuned': [
            f"{part2_finetuned_results['accuracy']:.4f} ({part2_finetuned_results['accuracy']*100:.2f}%)",
            f"{part2_finetuned_results['precision']:.4f} ({part2_finetuned_results['precision']*100:.2f}%)",
            f"{part2_finetuned_results['recall']:.4f} ({part2_finetuned_results['recall']*100:.2f}%)",
            f"{part2_finetuned_results['f1']:.4f} ({part2_finetuned_results['f1']*100:.2f}%)",
            f"{part2_finetuned_results['loss']:.4f}"
        ],
        'Improvement': []
    }
    
    # Calculate improvements
    metrics = ['accuracy', 'precision', 'recall', 'f1']
    for metric in metrics:
        pretrained_val = pretrained_part2_results[metric]
        finetuned_val = part2_finetuned_results[metric]
        if pretrained_val > 0:
            improvement = ((finetuned_val - pretrained_val) / pretrained_val) * 100
            symbol = "📈" if improvement > 0 else "📉" if improvement < 0 else "➡️"
            comparison_data['Improvement'].append(f"{symbol} {improvement:+.2f}%")
        else:
            comparison_data['Improvement'].append("N/A")
    
    # Loss improvement (lower is better)
    loss_improvement = ((pretrained_part2_results['loss'] - part2_finetuned_results['loss']) / pretrained_part2_results['loss']) * 100
    loss_symbol = "📈" if loss_improvement > 0 else "📉" if loss_improvement < 0 else "➡️"
    comparison_data['Improvement'].append(f"{loss_symbol} {loss_improvement:+.2f}%")
    
    # Display comparison
    df_comparison = pd.DataFrame(comparison_data)
    print(df_comparison.to_string(index=False))
    
    # Overall assessment
    f1_improvement = part2_finetuned_results['f1'] - pretrained_part2_results['f1']
    acc_improvement = part2_finetuned_results['accuracy'] - pretrained_part2_results['accuracy']
    
    print(f"\nOVERALL ASSESSMENT:")
    if f1_improvement > 0.01 and acc_improvement > 0.01:
        print("🎉 Fine-tuning on Part 2 dataset shows SIGNIFICANT improvement!")
    elif f1_improvement > 0 and acc_improvement > 0:
        print("✅ Fine-tuning on Part 2 dataset shows improvement.")
    elif abs(f1_improvement) < 0.01 and abs(acc_improvement) < 0.01:
        print("➡️ Fine-tuning shows similar performance to pretrained model.")
    else:
        print("⚠️ Fine-tuning may have reduced performance.")
        print("   Consider adjusting hyperparameters or checking data quality.")
    
    # Save comparison
    if os.path.isdir(PART2_FINETUNED_DIR):
        try:
            comparison_summary = {
                'dataset_source': dataset_source,
                'evaluation_samples': EVAL_SAMPLE_SIZE,
                'pretrained_baseline_results': pretrained_part2_results,
                'part2_finetuned_results': part2_finetuned_results,
                'improvements': {
                    'accuracy': acc_improvement,
                    'f1': f1_improvement,
                    'precision': part2_finetuned_results['precision'] - pretrained_part2_results['precision'],
                    'recall': part2_finetuned_results['recall'] - pretrained_part2_results['recall']
                },
                'column_mapping': column_info
            }
            
            filepath = os.path.join(PART2_FINETUNED_DIR, 'part2_comparison.json')
            with open(filepath, 'w') as f:
                json.dump(comparison_summary, f, indent=2)
            print(f"\n💾 Comparison results saved to: {filepath}")
        except Exception as e:
            print(f"Could not save comparison: {e}")
            
elif tokenized_part2 is None:
    print("❌ No Part 2 dataset loaded for comparison")
    print("Update PART2_DATASET_ID or PART2_LOCAL_PATH in the configuration cell")
else:
    print("⚠️ Incomplete results for comparison")
    if not pretrained_part2_results:
        print("Missing pretrained model results on Part 2 dataset")
    if not part2_finetuned_results:
        print("Missing Part 2 fine-tuned model results (set DO_PART2_TRAIN = True first)")

## Part 2 Summary

This notebook completed:

✅ **Independent Dataset Processing**: Loaded and processed Part 2 dataset independently  
✅ **Pretrained Baseline**: Tested original pretrained model on Part 2 dataset  
✅ **Part 2 Fine-tuning**: Fine-tuned pretrained model specifically on Part 2 dataset  
✅ **Performance Comparison**: Compared pretrained vs Part 2 fine-tuned performance  
✅ **Comprehensive Evaluation**: Generated detailed metrics, visualizations, and analysis  

**Workflow**:
1. **Load Part 2 Dataset** → Auto-detect columns and normalize labels
2. **Baseline Test** → Evaluate pretrained model on Part 2 data
3. **Fine-tune on Part 2** → Train pretrained model on Part 2 dataset
4. **Compare Results** → Analyze improvement from fine-tuning

**Key Features**:
- **Independent operation**: No dependency on Part 1 results
- **Auto-detection**: Automatically finds text and label columns
- **Label normalization**: Converts various label formats to binary 0/1
- **Class balancing**: Uses weighted loss for imbalanced datasets
- **Comprehensive metrics**: Accuracy, precision, recall, F1, ROC curves
- **Visual analysis**: Confusion matrices, ROC/PR curves, metrics tables

**Usage Instructions**:
1. Update `PART2_DATASET_ID` or `PART2_LOCAL_PATH` in configuration cell
2. Run cells to get pretrained baseline results
3. Set `DO_PART2_TRAIN = True` to fine-tune on Part 2 dataset
4. Compare pretrained vs fine-tuned performance

**Output Files**:
- `./models/hf_deberta_part2_finetuned/`: Fine-tuned model on Part 2 data
- `part2_finetuned_results.json`: Detailed fine-tuned model results
- `part2_comparison.json`: Performance comparison results
- `part2_train_log.json`: Training logs and metrics